# MoMA Collection - Exploratory Data Analysis

This notebook explores the Museum of Modern Art (MoMA) collection dataset to understand diversity patterns in representation.

## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 7)

artworks = pd.read_csv('../data/moma-collection/Artworks.csv')
artists = pd.read_csv('../data/moma-collection/Artists.csv')

print(f"Artworks: {len(artworks):,} records")
print(f"Artists: {len(artists):,} records")

## Data Cleaning

In [ ]:
# Clean Artists dataset
artists = artists.drop(columns=['Wiki QID', 'ULAN'])
artists_clean = artists.dropna(subset=['ArtistBio', 'Nationality', 'Gender', 'BeginDate'])

# Clean Artworks dataset
columns_to_drop = [
    'Dimensions', 'Height (cm)', 'Width (cm)', 'Depth (cm)',
    'Circumference (cm)', 'Diameter (cm)', 'Length (cm)',
    'Weight (kg)', 'Seat Height (cm)', 'Duration (sec.)',
    'AccessionNumber', 'CreditLine', 'Cataloged', 'OnView',
    'URL', 'ImageURL'
]
artworks = artworks.drop(columns=columns_to_drop)

key_artwork_columns = [
    'Title', 'Artist', 'ConstituentID', 'ArtistBio',
    'Nationality', 'BeginDate', 'EndDate', 'Gender',
    'Date', 'Medium', 'Classification', 'Department'
]
artworks_clean = artworks.dropna(subset=key_artwork_columns)

# Add temporal columns
artworks_clean['Date_numeric'] = pd.to_numeric(artworks_clean['Date'], errors='coerce')
artworks_clean = artworks_clean.dropna(subset=['Date_numeric'])
artworks_clean['Decade'] = (artworks_clean['Date_numeric'] // 10 * 10).astype(int)

print(f"Artists: {len(artists_clean):,} (cleaned)")
print(f"Artworks: {len(artworks_clean):,} (cleaned)")

## Gender Distribution

In [ ]:
gender_counts = artists_clean['Gender'].value_counts()
for gender, count in gender_counts.items():
    print(f"{gender}: {count:,} ({count / len(artists_clean) * 100:.1f}%)")

plt.figure(figsize=(8, 6))
gender_counts.plot(kind='bar', color=['skyblue', 'lightcoral', 'lightgreen'])
plt.title('Artist Gender Distribution')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Geographic Diversity

In [ ]:
nationality_stats = artworks_clean['Nationality'].value_counts()
print(f"Total unique nationalities: {artworks_clean['Nationality'].nunique()}")
print()
for nationality, count in nationality_stats.head(15).items():
    print(f"{nationality}: {count:,} ({count / len(artworks_clean) * 100:.1f}%)")

herfindahl = ((nationality_stats / nationality_stats.sum()) ** 2).sum()
print(f"\nHerfindahl Index: {herfindahl:.4f} (0 = perfect diversity, 1 = complete concentration)")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

nationality_stats.head(15).plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Top 15 Nationalities')
axes[0].set_xlabel('Number of Artworks')

cumsum = nationality_stats.cumsum() / nationality_stats.sum() * 100
axes[1].plot(range(len(cumsum.head(50))), cumsum.head(50).values, marker='o', linewidth=2, color='darkblue')
axes[1].axhline(y=80, color='red', linestyle='--', label='80% threshold')
axes[1].set_title('Cumulative Distribution (Top 50 Nationalities)')
axes[1].set_xlabel('Number of Nationalities')
axes[1].set_ylabel('Cumulative %')
axes[1].legend()

plt.tight_layout()
plt.show()

## Department Distribution

In [ ]:
dept_counts = artworks_clean['Department'].value_counts()
for dept, count in dept_counts.items():
    print(f"{dept}: {count:,} ({count / len(artworks_clean) * 100:.1f}%)")

plt.figure(figsize=(12, 6))
dept_counts.plot(kind='bar', color='darkgreen')
plt.title('Artworks by Department')
plt.xlabel('Department')
plt.ylabel('Number of Artworks')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Classification Analysis

In [ ]:
classifications = artworks_clean['Classification'].value_counts()
print(f"Total unique classifications: {len(classifications)}")
print()
for classification, count in classifications.items():
    print(f"{classification}: {count:,} ({count / len(artworks_clean) * 100:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 6))
classifications.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Artworks by Classification')
ax.set_xlabel('Classification')
ax.set_ylabel('Number of Artworks')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Medium Analysis

In [ ]:
top_mediums = artworks_clean['Medium'].value_counts().head(20)
print(f"Total unique mediums: {artworks_clean['Medium'].nunique()}")
print()
for medium, count in top_mediums.items():
    print(f"{medium}: {count:,} ({count / len(artworks_clean) * 100:.1f}%)")

print(f"\nTop 20 mediums represent {top_mediums.sum() / len(artworks_clean) * 100:.1f}% of collection")

fig, ax = plt.subplots(figsize=(12, 8))
top_mediums.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Mediums in MoMA Collection')
ax.set_xlabel('Number of Artworks')
ax.set_ylabel('Medium')
plt.tight_layout()
plt.show()

## Temporal Analysis

In [ ]:
decade_counts = artworks_clean['Decade'].value_counts().sort_index()
print(f"Temporal coverage: {artworks_clean['Date_numeric'].min():.0f} - {artworks_clean['Date_numeric'].max():.0f}")

plt.figure(figsize=(14, 6))
decade_counts.plot(kind='bar', color='orange')
plt.title('Artworks by Decade')
plt.xlabel('Decade')
plt.ylabel('Number of Artworks')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Medium Trends Over Time

In [ ]:
top_5_mediums = artworks_clean['Medium'].value_counts().head(5).index

fig, ax = plt.subplots(figsize=(14, 6))
for medium in top_5_mediums:
    medium_decade = artworks_clean[artworks_clean['Medium'] == medium].groupby('Decade').size()
    ax.plot(medium_decade.index, medium_decade.values, marker='o', label=medium, linewidth=2)

ax.set_title('Top 5 Mediums Over Time')
ax.set_xlabel('Decade')
ax.set_ylabel('Number of Artworks')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Artist Prolificacy

In [ ]:
works_per_artist = artworks_clean.groupby('Artist').size().reset_index(name='Works_Count')
works_per_artist = works_per_artist.sort_values('Works_Count', ascending=False)

print(f"Mean: {works_per_artist['Works_Count'].mean():.1f} | Median: {works_per_artist['Works_Count'].median():.0f} | Max: {works_per_artist['Works_Count'].max()}")
print()
print("Top 15 Most Prolific Artists:")
for _, row in works_per_artist.head(15).iterrows():
    print(f"{row['Artist']}: {row['Works_Count']:,} works")

top_100_contribution = works_per_artist.head(100)['Works_Count'].sum()
print(f"\nTop 1% of artists create {top_100_contribution / len(artworks_clean) * 100:.1f}% of artworks")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(works_per_artist['Works_Count'], bins=50, color='lightblue', edgecolor='black')
axes[0].set_title('Distribution of Works per Artist')
axes[0].set_xlabel('Number of Works')
axes[0].set_ylabel('Number of Artists')
axes[0].set_yscale('log')

top_artists = works_per_artist.head(15)
axes[1].barh(range(len(top_artists)), top_artists['Works_Count'], color='seagreen')
axes[1].set_yticks(range(len(top_artists)))
axes[1].set_yticklabels(top_artists['Artist'], fontsize=9)
axes[1].set_title('Top 15 Most Prolific Artists')
axes[1].set_xlabel('Number of Works')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Summary

In [ ]:
print(f"Artists: {len(artists_clean):,} | Artworks: {len(artworks_clean):,}")
print(f"Gender: {gender_counts['male'] / len(artists_clean) * 100:.1f}% male vs {gender_counts['female'] / len(artists_clean) * 100:.1f}% female")
print(f"Geographic concentration: Top 5 nationalities = {nationality_stats.head(5).sum() / len(artworks_clean) * 100:.1f}% of artworks")
print(f"Department focus: {dept_counts.index[0]} = {dept_counts.iloc[0] / len(artworks_clean) * 100:.1f}% of collection")
print(f"Temporal span: {artworks_clean['Date_numeric'].max() - artworks_clean['Date_numeric'].min():.0f} years")